# 시크릿 이력 정리와 예방

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.


## 준비

예제 문자열은 실제 서비스에서 사용할 수 없는 더미 자격 증명입니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 실제 인증에 사용할 수 없는 더미 문자열만 사용합니다.
DUMMY = "DEMO-ONLY-NOT-A-REAL-SECRET"
remote = root / "secret-remote.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
repo = root / "secret-demo"
git("clone", remote.as_posix(), repo.as_posix())
configure(repo)
(repo / "app.py").write_text('print("safe app")\n', encoding="utf-8")
git("add", "app.py", cwd=repo)
git("commit", "-m", "feat: 안전한 앱 시작", cwd=repo)
git("push", "-u", "origin", "main", cwd=repo)
print("더미만 사용:", DUMMY)

더미만 사용: DEMO-ONLY-NOT-A-REAL-SECRET


## 기본 실행

추적 전에는 .gitignore와 staged diff 검사를 함께 적용합니다.

In [3]:
# .env를 추적하기 전에 무시 규칙이 적용되는지 확인합니다.
(repo / ".gitignore").write_text(".env\n", encoding="utf-8")
(repo / ".env").write_text(f"DEMO_API_KEY={DUMMY}\n", encoding="utf-8")
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 환경 파일 제외", cwd=repo)
print(git("check-ignore", "-v", ".env", cwd=repo))
print("색인에 없음:", git("ls-files", ".env", cwd=repo) == "")

.gitignore:1:.env	.env
색인에 없음: True


## 값 하나를 바꿔 검사

안전한 설정 예시는 커밋할 수 있지만 더미 키 패턴이 있으면 검사를 실패로 판정합니다.

In [4]:
# staged diff는 이번 커밋에 실제로 포함될 내용만 보여 줍니다.
(repo / "settings.example").write_text("DEMO_API_KEY=replace-me\n", encoding="utf-8")
git("add", "settings.example", cwd=repo)
staged = git("diff", "--cached", "--", "settings.example", cwd=repo)
contains_dummy = DUMMY in staged
print("더미 자격 증명 발견:", contains_dummy)
print(staged)
assert not contains_dummy
git("commit", "-m", "docs: 설정 예시 추가", cwd=repo)
print("안전한 설정 예시 커밋 완료")

더미 자격 증명 발견: False
diff --git a/settings.example b/settings.example
new file mode 100644
index 0000000..02e3d73
--- /dev/null
+++ b/settings.example
@@ -0,0 +1 @@
+DEMO_API_KEY=replace-me
안전한 설정 예시 커밋 완료


## 실패 재현

파일을 다음 커밋에서 삭제해도 이전 커밋 객체에는 내용이 남습니다.

In [5]:
# 가상 키 파일을 다음 커밋에서 삭제해도 과거 커밋에는 값이 남습니다.
git("check-ignore", "-q", ".env", cwd=repo)
git("add", "-f", ".env", cwd=repo)
git("commit", "-m", "test: 더미 키를 잘못 기록", cwd=repo)
leak_commit = git("rev-parse", "HEAD", cwd=repo)
git("rm", ".env", cwd=repo)
git("commit", "-m", "fix: 환경 파일 삭제", cwd=repo)
git("push", cwd=repo)
historic = git("show", f"{leak_commit}:.env", cwd=repo)
print("현재 파일 존재:", (repo / ".env").exists())
print("이전 커밋 내용:", historic.strip())
assert DUMMY in historic

현재 파일 존재: False
이전 커밋 내용: DEMO_API_KEY=DEMO-ONLY-NOT-A-REAL-SECRET


## 수정

격리된 예제 저장소의 모든 참조에서 .env 경로를 제거한 뒤 원격을 강제로 갱신합니다.

In [6]:
# 격리된 원격의 모든 참조를 갱신하고 새 복제본에서 경로와 값을 검사합니다.
clean = root / "clean-history"
git("clone", "--mirror", remote.as_posix(), clean.as_posix())
result = subprocess.run(
    ["git", "filter-repo", "--force", "--path", ".env", "--invert-paths"],
    cwd=clean, text=True, capture_output=True, check=True,
)
git("remote", "add", "origin", remote.as_posix(), cwd=clean)
git("push", "--force", "--mirror", cwd=clean)
objects = git("rev-list", "--objects", "--all", cwd=clean)
value_hits = git("log", "-S", DUMMY, "--all", "--format=%H", cwd=clean)
print("정리 후 .env 경로 있음:", any(line.endswith(" .env") for line in objects.splitlines()))
print("정리 후 더미 값 커밋 수:", len(value_hits.splitlines()) if value_hits else 0)
verify = root / "verify-remote"
git("clone", "--mirror", remote.as_posix(), verify.as_posix())
remote_objects = git("rev-list", "--objects", "--all", cwd=verify)
remote_hits = git("log", "-S", DUMMY, "--all", "--format=%H", cwd=verify)
print("원격 재검사 값 커밋 수:", len(remote_hits.splitlines()) if remote_hits else 0)
assert not any(line.endswith(" .env") for line in objects.splitlines())
assert not any(line.endswith(" .env") for line in remote_objects.splitlines())
assert not value_hits and not remote_hits

정리 후 .env 경로 있음: False
정리 후 더미 값 커밋 수: 0
원격 재검사 값 커밋 수: 0
